# 06 — SISAM Air Quality Data Cleaning

Reads SISAM/CPTEC reanalysis air quality CSV files for 5 cities
across 2022–2024, cleans and standardizes to match project schema,
and saves to `data/processed/sisam_clean.parquet`.

Input: 5 CSV files in `data/raw/SISAM/`
Output: one parquet file conforming to SCHEMA.md

In [12]:
import pandas as pd
from pathlib import Path

ROOT = Path().resolve()
RAW = ROOT / "data" / "raw" / "SISAM"
PROCESSED = ROOT / "data" / "processed"

print(ROOT)
print(RAW)
print("Exists:", RAW.exists())

D:\projects\air_quality_brazil
D:\projects\air_quality_brazil\data\raw\SISAM
Exists: True


In [13]:
# List all files and confirm what we have
all_files = sorted(RAW.glob("*.csv"))

print(f"Total files found: {len(all_files)}\n")

for f in all_files:
    rows = sum(1 for _ in open(f, encoding="utf-8-sig")) - 1
    df_peek = pd.read_csv(f, encoding="utf-8-sig", nrows=1)
    city = df_peek["municipio"].iloc[0]
    print(f"{f.name[:2]}  {rows} rows  {city}")

Total files found: 5

AM  1096 rows  MANAUS
BH  1096 rows  BELO HORIZONTE
DF  1096 rows  BRASÍLIA
PA  1096 rows  PORTO ALEGRE
SP  1096 rows  SÃO PAULO


In [17]:
# City name mapping from SISAM Portuguese uppercase to project standard
city_map = {
    "MANAUS": "Manaus",
    "BELO HORIZONTE": "Belo Horizonte",
    "BRASÍLIA": "Brasília",
    "PORTO ALEGRE": "Porto Alegre",
    "SÃO PAULO": "São Paulo",
}

def read_sisam_file(filepath):
    """
    Reads a single SISAM reanalysis CSV file and returns a cleaned DataFrame.

    SISAM files use UTF-8 with BOM encoding, DD/MM/YYYY date format,
    and contain daily air quality reanalysis for one municipality.
    """

    # utf-8-sig handles the BOM character present in some SISAM exports
    df = pd.read_csv(filepath, encoding="utf-8-sig", quotechar="'")

    # Strip single quotes and BOM characters from municipio values
    # SISAM files have quoted values and occasional mid-file BOM characters
    df["municipio"] = (
        df["municipio"]
        .str.replace("'", "", regex=False)   # remove single quotes
        .str.replace("\ufeff", "", regex=False)  # remove BOM character
        .str.strip()
    )

    # Parse date from DD/MM/YYYY string to datetime
    df["date"] = pd.to_datetime(df["data"], format="%d/%m/%Y")

    # Add UTC timestamp at midnight to match INMET schema
    df["timestamp"] = df["date"].dt.tz_localize("UTC")

    # Add year column for grouping
    df["year"] = df["date"].dt.year

    # Standardize city name to project convention
    df["city"] = df["municipio"].str.strip().map(city_map)

    # Rename pollutant columns to cleaner names
    df = df.rename(columns={
        "pm2_5_reanalise": "pm25_ugm3",   # PM2.5 in μg/m³
        "pm10_reanalise":  "pm10_ugm3",   # PM10 in μg/m³
        "o3_reanalise":    "o3_ugm3",     # Ozone in μg/m³
        "co_reanalise":    "co_mgm3",     # CO in mg/m³
        "no2_reanalise":   "no2_ugm3",    # NO2 in μg/m³
    })

    # Select and order final columns
    df = df[[
        "timestamp", "date", "city", "year",
        "pm25_ugm3", "pm10_ugm3", "o3_ugm3", "co_mgm3", "no2_ugm3",
    ]]

    return df


# Load and concatenate all 5 files
dfs = []
for filepath in sorted(RAW.glob("*.csv")):
    try:
        df = read_sisam_file(filepath)
        dfs.append(df)
    except Exception as e:
        print(f"ERROR in {filepath.name}: {e}")

df_sisam = pd.concat(dfs, ignore_index=True)

# Sort by city and date — data arrives newest-first, we want chronological
df_sisam = df_sisam.sort_values(["city", "date"]).reset_index(drop=True)

print(f"Shape: {df_sisam.shape}")
print(f"Date range: {df_sisam['date'].min()} → {df_sisam['date'].max()}")
print(f"Cities: {sorted(df_sisam['city'].unique())}")
print(f"\nNulls:")
print(df_sisam.isnull().sum())
print(f"\nSample:")
print(df_sisam.head(3).to_string(index=False))

Shape: (5480, 9)
Date range: 2022-01-01 00:00:00 → 2024-12-31 00:00:00
Cities: ['Belo Horizonte', 'Brasília', 'Manaus', 'Porto Alegre', 'São Paulo']

Nulls:
timestamp    0
date         0
city         0
year         0
pm25_ugm3    0
pm10_ugm3    0
o3_ugm3      0
co_mgm3      0
no2_ugm3     0
dtype: int64

Sample:
                timestamp       date           city  year  pm25_ugm3  pm10_ugm3  o3_ugm3  co_mgm3  no2_ugm3
2022-01-01 00:00:00+00:00 2022-01-01 Belo Horizonte  2022      11.27      16.38    31.24     0.10      5.27
2022-01-02 00:00:00+00:00 2022-01-02 Belo Horizonte  2022      13.60      19.78    29.82     0.13      6.15
2022-01-03 00:00:00+00:00 2022-01-03 Belo Horizonte  2022      13.81      19.91    29.85     0.12      5.29


In [18]:
# Save to processed/
output_path = PROCESSED / "sisam_clean.parquet"
df_sisam.to_parquet(output_path, index=False)

print(f"Saved to {output_path}")
print(f"Shape: {df_sisam.shape}")
print(f"File size: {output_path.stat().st_size / 1024 / 1024:.2f} MB")

Saved to D:\projects\air_quality_brazil\data\processed\sisam_clean.parquet
Shape: (5480, 9)
File size: 0.11 MB


In [19]:
# Reload and verify
df_check = pd.read_parquet(PROCESSED / "sisam_clean.parquet")

print(f"Shape: {df_check.shape}")
print(f"\nDtypes:")
print(df_check.dtypes)
print(f"\nRows per city:")
print(df_check["city"].value_counts().sort_index())
print(f"\nNulls: {df_check.isnull().sum().sum()}")

Shape: (5480, 9)

Dtypes:
timestamp    datetime64[us, UTC]
date              datetime64[us]
city                         str
year                       int32
pm25_ugm3                float64
pm10_ugm3                float64
o3_ugm3                  float64
co_mgm3                  float64
no2_ugm3                 float64
dtype: object

Rows per city:
city
Belo Horizonte    1096
Brasília          1096
Manaus            1096
Porto Alegre      1096
São Paulo         1096
Name: count, dtype: int64

Nulls: 0


# Cleaning Summary — SISAM air quality

## Input
- 5 CSV files from SISAM/CPTEC reanalysis system
- One file per city: Manaus, Belo Horizonte, Brasília, Porto Alegre, São Paulo
- Daily reanalysis data (not ground station measurements)

## Issues fixed
- BOM character (`\ufeff`) embedded mid-file on some rows
- Single quotes around column names and values in CSV export
- City names stripped and mapped to project standard

## Output
- 5,480 rows × 9 columns (5 cities × 1,096 days)
- Date range: 2022-01-01 → 2024-12-31
- Zero nulls across all columns
- Saved to data/processed/sisam_clean.parquet (0.11 MB)

## Columns
- pm25_ugm3 — PM2.5 in μg/m³
- pm10_ugm3 — PM10 in μg/m³
- o3_ugm3   — Ozone in μg/m³
- co_mgm3   — CO in mg/m³
- no2_ugm3  — NO2 in μg/m³

## Limitation
- Reanalysis data, not direct ground measurements
- Coverage ends 2024-12-31 — does not match INMET/hotspots 2025 data
- 2025 air quality analysis not possible with this dataset